# Mixed layer tracer budgets in ACCESS-OM2 - background theory and diagnostics

This notebook contains a short overview of the theory and diagnostics used within this repository. More details can be found in,

Holmes, Malan and Bladwell, Accurately diagnosing mixed layer tracer budgets in a global ocean model, in preparation.

This notebook contains no code.

## The ACCESS-OM2/MOM5 "Eulerian" heat budget

The budget for temperature within a single grid-cell of a finite-volume global ocean model can be formulated as:

\begin{equation}
    \frac{\partial}{\partial t}\left(\int_R \rho_0 C dV\right) = -\oint_{\partial R} \left[\rho_0 C (\mathbf{v} - \mathbf{v}^{(b)}) + \mathbf{J}\right]\cdot\mathbf{\hat{n}}\,d\mathcal{S},
\end{equation}

where $R$ represents the grid cell region, $C$ is the tracer concentration $=C_p \Theta$ for temperature, $\mathbf{\hat{n}}$ is the outward normal vector on the boundary surface $\partial R$, $d\mathcal{S}$ is the area element on that boundary, $\mathbf{v}$ is the fluid velocity, $\mathbf{v}^{(b)}$ is the velocity of the boundary and $\mathbf{J}$ represents tracer fluxes across the boundary surface associated with sub-grid scale parameterizations (such as vertical mixing) and boundary fluxes (such as air-sea tracer fluxes).

The equivalent equation in terms of MOM5 diagnostics (per unit area) is given by:
\begin{align}
  \textit{temp\_tendency} = &\textit{temp\_advection} + \\ &\quad +
  \textit{temp\_submeso} \\ &\quad + \textit{temp\_vdiffuse\_diff\_cbt}  + \textit{temp\_nonlocal\_KPP} \\ &\quad + \textit{sw\_heat} +
  \textit{temp\_rivermix} + \textit{temp\_vdiffuse\_sbc} + \textit{sfc\_hflux\_pme} \\
  & \quad + \textit{frazil\_3d}\\ 
   & \quad + \textit{temp\_vdiffuse\_k33} + \textit{neutral\_diffusion\_temp}\\
   & \quad + \textit{neutral\_gm\_temp} \\
   & \quad + \textit{mixdownslope\_temp} + \textit{temp\_sigma\_diff} + \textit{temp\_eta\_smooth}
\end{align}

All terms are in units of Wm$^{-2}$ - i.e. the tendency of the heat content within each grid cell per unit area, $\rho_0 C_p\Theta \Delta z$, where $\Delta z$ is the time variable grid cell thickness, $\rho_0=1035$kgm$^{-3}$ is the reference density, $C_p=3992.10322329649$Jkg$^{-1}$$^\circ$C$^{-1}$ is the specific heat and $\Theta$ is Conservative Temperature.

- temp\_tendency is the tendency term
- temp\_advection is the convergence of the three-dimensional resolved advection (this can be split into components by taking the convergence of the temp\_xflux\_adv, temp\_yflux\_adv and temp\_zflux\_adv terms). Note that this is equivalent to
\begin{equation}
-\oint_{\partial R - \partial\eta} \rho_0 C (\mathbf{v} - \mathbf{v}^{(b)})\cdot\mathbf{\hat{n}}\,d\mathcal{S},
\end{equation}
i.e. it does not include the dia-surface motion across the free-surface ($\partial\eta$), which is instead captured by $\textit{sfc\_hflux\_pme}$, equal to $-\oint_{\partial\eta} \rho_0 C (\mathbf{v} - \mathbf{v}^{(b)})\cdot\mathbf{\hat{n}}\,d\mathcal{S}$
- temp\_submeso is the convergence of the three-dimensional parameterized submesoscale advection (pretty small).
- temp\_vdiffuse\_diff\_cbt and temp\_nonlocal\_KPP are the vertical mixing terms.
- The next line contains all of the surface heat flux terms. Note that sw\_heat is a three-dimensional term that *redistributes* the impact of SW radiation from the surface layer into the interior (i.e. it is negative in the surface layer and positive below, summing to zero). temp\_rivermix, a term that mixes vertically in regions of river runoff, is also three-dimensional as the impact of river runoff is spread over a few layers (4 I think). The other terms are two-dimensional (only non-zero in the surface layer).
- frazil\_3d is the formation of frazil ice
- temp\_vdiffuse\_k33 and neutral\_diffusion\_temp are parameterized along-isopycnal mixing (might not be on in all configurations, e.g. ACCESS-OM2-01).
- neutral\_gm\_temp is parameterization advection by mesoscale eddies
- The last line includes some miscellaneous mixing terms (all pretty small, and not all active depending on configuration).

Also see https://github.com/COSIMA/access-om2/issues/139#issuecomment-639278547 for a discussion of the surface heat flux terms in ACCESS-OM2/CM2. See `Testing_and_Checks.ipynb` for a check of the closure of this budget.

## The mixed layer temperature budget 

The mixed layer depth is defined as the depth at which the buoyancy difference to the surface layer is $0.0003$ms$^{-2}$, corresponding to a density difference of $0.031$kgm$^{-3}$. 
$0.03$kgm$^{-3}$ is a widely used value in the literature. We define the mixed layer in a continuous sense, such that the mixed layer base can lie between two grid cells (linear interpolation), with a known fractional contribution of the bottom grid cell to the mixed layer. The mixed layer within a given model column is defined by the region $\partial R_H$.

We are most interested in the tracer concentration averaged over the mixed layer volume, rather than the total mixed layer tracer content.
We define this (for temperature) as,
\begin{equation}
    \Theta_H \equiv \frac{1}{H}\int_{-H_z}^\eta \Theta dz,
\end{equation}
where $\eta$ is the free-surface height and $H_z$ is the depth of the mixed layer (from $z=0$), such that the total mixed layer depth is $H=\eta+H_z$.

A bunch of maths, shown elsewhere (paper in preparation), shows that $\Theta_H$ obeys the budget equation,

\begin{align}
        \frac{\partial \Theta_H}{\partial t}&= \quad\quad &\quad\quad\text{tendency}\\ &\quad-\frac{1}{AH}\oint_{\partial R_H - \partial\eta} \left(\Theta-\Theta_H\right) \mathbf{v}\cdot\mathbf{\hat{n}}\,d\mathcal{S}-\frac{Q_{\text{L}}}{\rho_0 H}\quad\quad &\quad\quad\text{advection (+ eddy processes)} \\
        &\quad+\frac{Q_\text{net}}{\rho_0 H}-\frac{\Theta_a - \Theta_H}{\rho_0H}Q_m\quad\quad &\quad\quad\text{surface fluxes} \\
        &\quad-\frac{Q_{\text{SWP}}}{\rho_0 H}\quad\quad &\quad\quad\text{shortwave penetration} \\
        &\quad-\frac{Q_\text{mix}}{\rho_0 H}\quad\quad &\quad\quad\text{vertical mixing} \\
        &\quad-\frac{\Theta_{\text{ent}}-\Theta_H}{H}\frac{\partial H_z}{\partial t}\quad\quad &\quad\quad\text{entrainment}
\end{align}
where $A$ is the area of the grid cell and other terms are described below, in order.

#### Tendency (LHS): 
The first line is the tendency term. Note that this term is not a diagnostic in MOM5 (it is not temp\_tendency, which is the tendency of the total heat content of the grid cells making up the mixed layer, $R_{\Sigma G}$ in the paper) and needs to be computed offline (from snapshots or time-averages, depending on whether standard or hat-averaging is used on the budget diagnostics) using a difference of mixed layer temperature diagnostics ($\textit{temp\_in\_mld}$, equal to $\Theta_H\rho_0$). This is done in the function `compute_tendency_entrainment` in the budget processing scripts.

#### Advection/eddy processes: 
The second line represents advection and eddy driven processes. This term is effectively equal to $\textit{temp\_advection} + \textit{temp\_submeso} + \textit{temp\_vdiffuse\_k33} + \textit{neutral\_diffusion\_temp} + \textit{neutral\_gm\_temp}$ divided by $C_p\rho_0 H$, except that since $H$ is time-varying, this division needs to be done at every time-step. This is done by the new diagnostics $\textit{temp\_advection\_in\_mld}$ (and equivalent for the eddy terms), which are equal to $\textit{temp\_advection}/H$ summed over the mixed layer (so divide these by $\rho_0 C_p$ to get the form used in the above equation). 

However, we also note that more maths (see the appendix of the paper), shows that two additional correction terms,
\begin{equation}
\frac{\Theta_H}{H}\nabla\cdot\mathbf{U} + \frac{\Theta_{\text{ent}}}{H} w^{(s)}_H
\end{equation}
need to be added to $\textit{temp\_advection\_in\_mld}/\rho_0/C_p$ in order to get the advection term in the above equation. The corrections account for the fact that the advection diagnostic $\textit{temp\_advection}$ is computed in GVC coordinates, not Eulerian coordinates. $\nabla\cdot\mathbf{U}$ is the divergence of the vertically-integrated transport (throughout the whole ocean depth) and $w^{(s)}_H$ is the vertical velocity of the GVC coordinate at the base of the mixed layer. We compute these two terms separately. Note that both of these correction terms are dependent on the temperature scale (e.g. Kelvin vs. Celsius), and thus remove the dependence of the temperature scale in $\textit{temp\_advection\_in\_mld}$ (since the form of the advection term in the budget equation above depends only on temperature differences).

To compute the $\frac{\Theta_H}{H}\nabla\cdot\mathbf{U}$ term, we add three new diagnostics, $\textit{eta\_t\_tendency\_times\_temp\_in\_mld}$, $\textit{pme\_river\_times\_temp\_in\_mld}$ and $\textit{eta\_smoother\_times\_temp\_in\_mld}$, computing $\frac{\Theta_H}{H}\nabla\cdot\mathbf{U}$ by residual of the free-surface equation (see testing section below),

\begin{equation}
\frac{\partial \eta}{\partial t} = -\nabla\cdot\mathbf{U} + Q_m/\rho_0 + S_{\text{smoother}}
\end{equation}

To compute the $w^{(s)}_H$ dependent term, we compute $\Theta_{\text{ent}}$ by linearly interpolating the temperature to the mixed layer base, which is an approximation (although note that we show later on that this term is quite small). $w^{(s)}_H$ is computed for the $z^*$ vertical coordinates used in MOM5 via,

\begin{equation}
w_{H}^{(s)} = (1-\frac{H}{D+\eta})\frac{\partial \eta}{\partial t},
\end{equation}
where $D$ is the ocean depth. This term is contained within the new diagnostic $\textit{s\_surf\_ent\_temp}$. Note that for testing purposes we have also added new diagnostics $\textit{temp\_at\_mlb}$, which is $\Theta_{\text{ent}}$ and $\textit{eta\_t\_tendency\_times\_temp\_at\_mlb}$, which is equal to $\Theta_{\text{ent}}\frac{\partial\eta}{\partial t}/H$.

In summary, the advection term is given by the following combination of terms:

\begin{equation}
\frac{1}{\rho_0 C_p} \left[\textit{temp\_advection\_in\_mld}\right] + \textit{adv\_cor1} + \textit{adv\_cor2} + \frac{1}{\rho_0 C_p} \left[\textit{temp\_submeso\_in\_mld} + \textit{neutral\_diffusion\_in\_mld\_temp} + \textit{neutral\_gm\_in\_mld\_temp} + \textit{temp\_vdiffuse\_k33\_in\_mld}\right]
\end{equation}

where the advection corrections are,

\begin{align}
      \textit{adv\_cor1} &= \frac{1}{\rho_0}\left[-\textit{eta\_t\_tendency\_times\_temp\_in\_mld} + \textit{pme\_river\_times\_temp\_in\_mld} + \textit{eta\_smoother\_times\_temp\_in\_mld}\right] \\
      \textit{adv\_cor2} &= \frac{1}{\rho_0} \textit{s\_surf\_ent\_temp}
\end{align}

#### Surface fluxes: 
The third line represents surface fluxes, including surface mass fluxes and ice-ocean processes (like frazil formation). The only surface flux process that it doesn't include is shortwave redistribution. The second part of this term in the above equation accounts for the impact of surface mass fluxes, $Q_m$ (in kgs$^{-1}$), where $C_a$ is the tracer concentration of the added (or removed) surface mass. For temperature in MOM5, $C_a$ is equal to the temperature of the top grid cell. The $C_a$ dependent part of this term is $\textit{sfc\_hflux\_pme\_in\_mld}$. However, the $\Theta_H$ dependent portion,
\begin{equation}
\Theta_H Q_m/(\rho_0 H),
\end{equation} 
is a correction that needs to be computed separately. Again, due to the fact that $Q_m$, $\Theta_H$ and $H$ all vary in time, this term needs to be computed online and is captured by the new diagnostic $\textit{pme\_river\_times\_temp\_in\_mld}$. 

Note that for simplicity (and because its small) we include $\textit{temp\_eta\_smooth\_in\_mld}$ in this term as well. Furthermore, we note that because the free-surface smoothing term is also a mass-flux related term, it also needs a correction similar to that used above - ie. we subtract $\textit{eta\_smoother\_times\_temp\_in\_mld}$ from it. Thus, all the mass-flux related terms (advection, P-E+R and the SSH-smoother) all need these types of corrections.

In summary, the surface flux term is given by the following:

\begin{align}
&\frac{1}{\rho_0 C_p} \left[\textit{temp\_rivermix\_in\_mld} + \textit{temp\_vdiffuse\_sbc\_in\_mld} + \textit{frazil\_3d\_in\_mld}\right] \\
&+ \frac{1}{\rho_0 C_p}\textit{sfc\_hflux\_pme\_in\_mld} - \frac{1}{\rho_0} \textit{pme\_river\_times\_temp\_in\_mld} \\
&+ \frac{1}{\rho_0 C_p}\textit{temp\_eta\_smooth\_in\_mld\_cor}  - \frac{1}{\rho_0} \textit{eta\_smoother\_times\_temp\_in\_mld} 
\end{align}

Note that the contributions of latent, sensible, shortwave and longwave surface fluxes can also be analysed separately through the terms $\textit{swflx\_in\_mld}$, $\textit{lw\_heat\_in\_mld}$, $\textit{sens\_heat\_in\_mld}$, $\textit{evap\_heat\_in\_mld}$ (all divided by $\rho_0 C_p$).

#### Shortwave penetration: 
The fourth line represents shortwave penetration, simply equal to $\textit{sw\_heat\_in\_mld}/\rho_0/C_p$.

#### Vertical mixing: 
The fifth line represents vertical mixing at the base of the mixed layer, equal to $(\textit{temp\_vdiffuse\_diff\_cbt\_in\_mld}  + \textit{temp\_nonlocal\_KPP\_in\_mld})/\rho_0/C_p$.

#### Entrainment: 
The last line represents entrainment, where $\Theta_\text{ent}$ is the temperature of the entrained water. Due to the fact that $\Theta_{\text{ent}}$ needs a interpolation step to be computed, we instead compute this entire term by residual of the budget above.

Note that the term $\textit{temp\_tendency\_in\_mld}$ is the sum of all the \textit{\_in\_mld terms}, which represents (apart from the 3 correction terms discussed above) the tendency in the heat content of the layer ignoring the extra heat content entering through entrainment. Hence (again ignoring the 3 corrections noted above) entrainment effectively represents the difference between $\textit{temp\_tendency\_in\_mld}$ and $\partial \Theta_H/\partial t$.

The bulk of the work in computing these grouped budget terms is captured by the functions `compute_corrections`, `mlt_budget_fixedh` and `compute_tendency_entrainment` (also see `bud_var_grps`) used in the budget processing scripts in this repository.

## Hat averaging

The following material comes from Bladwell et al. (2025, in prep.). Consider a variable $\xi(t)$ (e.g. mixed layer temperature at a single location), defined as a function of time. We will consider two ``epochs" defined by the time periods $t\in(t_1,t_1+\Delta t_1)$ and  $t\in(t_2,t_2+\Delta t_2)$. The standard average of the tendency of $\xi$ between $t_1$ and $t_2+\Delta t_2$ (i.e. over the entire period covered by standard tendency diagnostics), multiplied by the time gap between them, is given by:
\begin{equation}
(t_2+\Delta t_2 - t_1)\overline{\frac{\partial\xi}{\partial t}}^{t_1,t_2+\Delta t_2} \equiv \int_{t_1}^{t_2+\Delta t_2} \frac{\partial\xi}{\partial t} dt = \left[\xi(t_2+\Delta t_2) - \xi(t_1)\right]
\end{equation}
This corresponds to a difference in snapshots of $\xi$, and thus is not typically a quantity of interest.

Instead, we define the "hat average" operator between the two epochs as,
\begin{equation}
\hat{\frac{\partial\xi}{\partial t}}^{t_1,t_1+\Delta t_1,t_2,t_2+\Delta t_2} \equiv \int_{t_1}^{t_1+\Delta t_1} \frac{t-t_1}{\Delta t_1}\frac{\partial\xi}{\partial t} dt + \int_{t_1+\Delta t_1}^{t_2} \frac{\partial\xi}{\partial t} dt + \int_{t_2}^{t_2+\Delta t_2} \frac{t_2+\Delta t_2 - t}{\Delta t_2}\frac{\partial\xi}{\partial t} dt = \overline{\xi}^{t_2,t_2+\Delta t_2} - \overline{\xi}^{t_1,t_1+\Delta t_1}
\end{equation}
This corresponds to a "rising average" over the first epoch, and standard average between them, and a falling average over the second epoch (hence the "hat average"). Evidently, the hat average between the two epochs is the operation needed to relate the tendency $\partial\xi/\partial t$ to the difference between the values of $\xi$ averaged over the two epochs.

Note that the diagnostics output from MOM5 to form the hat averaging correspond to standard, rising and falling averages over a shorter, pre-defined time period (below daily) that does not usually correspond to the epoch differences of interest (e.g. differences between months). If our longer epoch of interest, say $t\in(t_1,t_1+\Delta t_1)$, consists of $N$ sections of shorter diagnostics of length $\Delta t$ (e.g. the month of January consists of $N=31$ sections of length $\Delta t=1$ day, with $\Delta t_1=N\Delta t$), then we can use the following formula's to compute the rising "difference" (i.e. the first term in the previous equation) over the longer (i.e. entire January) period from the rising averages over each day,
\begin{equation}
\int_{t_1}^{t_1+N\Delta t} \frac{t-t_1}{N\Delta t} \frac{\partial\xi}{\partial t}dt = \sum_{n=1}^N \frac{1}{N} \int_{t_1+(n-1)\Delta t}^{t_1+n\Delta t} \frac{t-(t_1+(n-1)\Delta t)}{\Delta t} \frac{\partial\xi}{\partial t} dt + \sum_{n=1}^N \frac{(n-1)}{N} \int_{t_1+(n-1)\Delta t}^{t_1+n\Delta t} \frac{\partial\xi}{\partial t} dt
\end{equation}
where the LHS represents the long rising difference over the period $t\in(t_1,t_1+\Delta t_1)$, the first term on the RHS is the sum of $1/N$ times the short rising difference over the short period plus $(n-1)/N$ times the short standard difference over the short period.
The long standard difference is trivially,
\begin{equation}
\int_{t_1}^{t_1+N\Delta t} \frac{\partial\xi}{\partial t}dt = \sum_{n=1}^N \int_{t_1+(n-1)\Delta t}^{t_1+n\Delta t} \frac{\partial\xi}{\partial t} dt
\end{equation}
the long falling difference can be computed as the difference between the two previous equations
\begin{equation}
\int_{t_1}^{t_1+N\Delta t} \frac{t_1 + N\Delta t - t}{N\Delta t} \frac{\partial\xi}{\partial t}dt = \int_{t_1}^{t_1+N\Delta t} \frac{\partial\xi}{\partial t}dt - \int_{t_1}^{t_1+N\Delta t} \frac{t-t_1}{N\Delta t} \frac{\partial\xi}{\partial t}dt
\end{equation}